## Langchain Expression Language (LCEL)
- [LCEL Tutorial YT](https://www.youtube.com/watch?v=NQWfvhw7OcI)
- [Langchain Doc](https://python.langchain.com/docs/tutorials/)

### Content
* What is LangChain Expression Language (LCEL)
* LCEL Example
* Why we should use it
* LCEL RAG (Retrieval Augmented Generation) Example

In [1]:
# !pip install langchain
# !pip install openai
# !pip install chromadb
# !pip install tiktoken

In [2]:
from pprint import pprint as pp

In [3]:
# check and setup environment variables and API keys for Langchain and OpenAI

import os
import dotenv

dotenv.load_dotenv()

# LANGCHAIN_PROJECT to set the project name for Langchain
print("LANGCHAIN_PROJECT:", os.environ.get("LANGCHAIN_PROJECT", "langchain-intro"))
# LANGCHAIN_TRACING to keep track of the execution
print("LANGCHAIN_TRACING:", os.environ.get("LANGCHAIN_TRACING_V2", "Not set"))
# LANGSMITH_ENDPOINT to set the endpoint for Langsmith
print("Langsmith Endpoint:", os.environ.get("LANGSMITH_ENDPOINT", "Not set"))
# OPENAI_API_KEY to set the API key for OpenAI
print("OpenAI API Key:", os.environ.get("OPENAI_API_KEY", "No API Key found")[:10])
# LANGCHAIN_API_KEY to set the API key for Langchain
print("Langchain API Key:", os.environ.get("LANGCHAIN_API_KEY", "No API Key found")[:10])

LANGCHAIN_PROJECT: langchain-intro
LANGCHAIN_TRACING: true
Langsmith Endpoint: https://api.smith.langchain.com
OpenAI API Key: sk-proj-Wj
Langchain API Key: lsv2_pt_70


In [4]:
from langchain_openai import ChatOpenAI
from langchain.prompts import ChatPromptTemplate
from langchain.schema.output_parser import StrOutputParser

In [5]:
# Initialize the ChatOpenAI model
model = ChatOpenAI()

# Define an output parser to interpret the model's response as a string
output_parser = StrOutputParser()

# Define a prompt template for generating a product description from a string of product notes

prompt = ChatPromptTemplate.from_template(
    "Create a lively and engaging product description with emojis based on these notes: \n"
    "{product_notes}"
)

In [6]:
prompt_value = prompt.invoke({"product_notes": "Multi color affordable mobile covers"})
prompt_value

ChatPromptValue(messages=[HumanMessage(content='Create a lively and engaging product description with emojis based on these notes: \nMulti color affordable mobile covers', additional_kwargs={}, response_metadata={})])

In [7]:
prompt_value.to_string()

'Human: Create a lively and engaging product description with emojis based on these notes: \nMulti color affordable mobile covers'

In [8]:
prompt_value.to_messages()

[HumanMessage(content='Create a lively and engaging product description with emojis based on these notes: \nMulti color affordable mobile covers', additional_kwargs={}, response_metadata={})]

In [9]:
model_output = model.invoke(prompt_value.to_messages())
model_output

AIMessage(content="Are you tired of your boring old phone case? Spice things up with our vibrant multi-color mobile covers! 🌈 Not only do they add a pop of personality to your device, but they're also super affordable, so you can switch up your style whenever you want! 💸📱 Say goodbye to bland and hello to fabulous with our eye-catching covers! Get yours today and let your phone shine in style! ✨ #PhoneGoals #ColorfulTech", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 94, 'prompt_tokens': 27, 'total_tokens': 121, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-BuOgXUNb8Q51eWJpZHmqmUayBD15O', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--44df07ec-be62-4a66-a705-610558380703

In [10]:
output_parser.invoke(model_output)

"Are you tired of your boring old phone case? Spice things up with our vibrant multi-color mobile covers! 🌈 Not only do they add a pop of personality to your device, but they're also super affordable, so you can switch up your style whenever you want! 💸📱 Say goodbye to bland and hello to fabulous with our eye-catching covers! Get yours today and let your phone shine in style! ✨ #PhoneGoals #ColorfulTech"

### Why use LCEL?

- [Here](https://python.langchain.com/docs/expression_language/#benefits-of-lcel)

In [11]:
chain = prompt | model | output_parser
chain

ChatPromptTemplate(input_variables=['product_notes'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['product_notes'], input_types={}, partial_variables={}, template='Create a lively and engaging product description with emojis based on these notes: \n{product_notes}'), additional_kwargs={})])
| ChatOpenAI(client=<openai.resources.chat.completions.completions.Completions object at 0x00000262686B7910>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x0000026268B2C390>, root_client=<openai.OpenAI object at 0x0000026268518F10>, root_async_client=<openai.AsyncOpenAI object at 0x0000026268B07ED0>, model_kwargs={}, openai_api_key=SecretStr('**********'))
| StrOutputParser()

In [12]:
product_description = chain.invoke(
    {"product_notes": "Multi color affordable mobile covers"}
)
pp(product_description)

('Introducing our vibrant and affordable collection of multi-color mobile '
 'covers! 🎨📱 These covers are perfect for adding a pop of color and '
 'personality to your phone while keeping it protected from everyday wear and '
 'tear. With a wide range of colors to choose from, you can mix and match to '
 'suit your mood or outfit. Made from high-quality materials, our mobile '
 'covers are durable and long-lasting. Say goodbye to boring phone cases and '
 'give your device a stylish upgrade with our multi-color mobile covers! Get '
 'yours today and stand out from the crowd! 🌈💃 #ColorfulTech #ProtectYourPhone '
 '#AffordableStyle')


In [13]:
for chunk in chain.stream({"product_notes": "Multi color affordable mobile covers"}):
    print(chunk, end="", flush=True)

Introducing our latest collection of multi-color mobile covers that are not only affordable but also super stylish! 🌈💸 

Add a pop of color to your everyday look with these vibrant and eye-catching designs that are sure to turn heads wherever you go. 📱💫 

Made from high-quality materials, these covers are not only durable but also provide excellent protection for your phone from scratches and bumps. 🔒💥

Choose from a variety of fun and exciting color combinations to match your mood and style - perfect for every fashionista on a budget! 💃🏼🎨 

Upgrade your phone game with our multi-color affordable mobile covers today and stand out in a crowd! 🙌📱 #ColorfulAndChic

### Batch description
- used to run all the invocation parallely, can save a lot of time

In [14]:
product_notes_list = [
    {"product_notes": "Multi-color affordable mobile covers"},
    {"product_notes": "Eco-friendly reusable water bottles"},
    {"product_notes": "Handcrafted wooden desk organizers"},
    {"product_notes": "High-performance running shoes"},
    {"product_notes": "Compact and lightweight travel backpack"},
    {"product_notes": "Organic and vegan skincare products"},
    {"product_notes": "Smart home automation system"},
    {"product_notes": "Energy-efficient LED lighting solutions"},
    {"product_notes": "Premium quality wireless headphones"},
    {"product_notes": "Durable and stylish kitchenware set"},
]

In [15]:
batch_descriptions = chain.batch(product_notes_list)
print(len(batch_descriptions))
pp(batch_descriptions[0])  # Print the first description


10
('Introducing our vibrant and budget-friendly mobile covers! 🌈📱 Keep your '
 'phone protected in style with our multi-color designs that will make your '
 'device stand out from the crowd. Made from high-quality materials, our '
 'covers are durable and offer maximum protection against everyday wear and '
 'tear. Choose from a wide range of colors and patterns to match your unique '
 'personality. Upgrade your phone game with our trendy mobile covers today! 💥 '
 '#PhoneFashion #AffordableStyle')


In [16]:
for desc in batch_descriptions:
    pp(desc)
    print("-" * 100)  # Separator for readability

('Introducing our vibrant and budget-friendly mobile covers! 🌈📱 Keep your '
 'phone protected in style with our multi-color designs that will make your '
 'device stand out from the crowd. Made from high-quality materials, our '
 'covers are durable and offer maximum protection against everyday wear and '
 'tear. Choose from a wide range of colors and patterns to match your unique '
 'personality. Upgrade your phone game with our trendy mobile covers today! 💥 '
 '#PhoneFashion #AffordableStyle')
----------------------------------------------------------------------------------------------------
('Introducing our eco-friendly reusable water bottles! 🌿💧 Not only are these '
 'sleek bottles helping to reduce waste and minimize single-use plastic, but '
 'they also keep your drinks refreshingly cool all day long. With a wide '
 "variety of colors and sizes to choose from, there's a perfect bottle for "
 'everyone. Take a stand for the planet and stay hydrated in style with our '
 'eco-frie

## LCEL RAG Example

In [17]:
docs = [
    "Climate Change Impact: Recent studies show that the average global temperature has risen by approximately 1 degree Celsius since the late 19th century. This change has led to significant melting of polar ice caps, resulting in rising sea levels. The impact on wildlife is profound, with many species struggling to adapt to rapidly changing habitats.",
    "AI in Healthcare: Artificial intelligence is revolutionizing healthcare with predictive analytics and personalized medicine. For instance, AI algorithms are being used to analyze medical imaging more accurately than traditional methods, enabling early detection of diseases like cancer. Personalized treatment plans based on genetic profiles are becoming increasingly common, tailoring healthcare to individual patient needs.",
    "Mars Exploration: NASA's Perseverance rover, which landed on Mars in February 2021, aims to explore the Jezero Crater. The mission's goals include searching for signs of ancient life and collecting samples of Martian rock and regolith (broken rock and soil) for potential return to Earth. This mission is a significant step in understanding Mars' habitability.",
    "Gene Editing: CRISPR-Cas9 technology has emerged as a groundbreaking tool in gene editing. It allows scientists to edit DNA sequences and modify gene function with high precision. Its potential applications range from treating genetic disorders to enhancing crop resilience. However, ethical debates arise around the possibilities of 'designer babies' and genetic privacy concerns.",
    "Cuisine and Culture: The Mediterranean diet, characterized by high consumption of fruits, vegetables, fish, and olive oil, reflects the culinary traditions of countries like Greece and Italy. It's not just about the ingredients but also about the communal aspect of food preparation and consumption, highlighting how cuisine is deeply intertwined with cultural identity and lifestyle.",
    "Social Media and Politics: The Cambridge Analytica scandal revealed how social media data could be used to influence political opinions. The company collected personal data from millions of Facebook users without consent and used it for political advertising. This event sparked a global discussion on data privacy and the ethical responsibilities of social media platforms.",
    "Renewable Energy Progress: Solar power technology has seen significant advancements, with the development of photovoltaic cells that can convert more than 22% of sunlight into electricity. Countries like Germany and China are leading in solar energy production, contributing to a global shift towards renewable energy sources to combat climate change and reduce reliance on fossil fuels.",
]

In [18]:
from langchain_openai import OpenAIEmbeddings
from langchain.prompts import ChatPromptTemplate
from langchain.schema.runnable import RunnableParallel, RunnablePassthrough
from langchain.vectorstores import Chroma

In [19]:
# Create a vector store from the documents using OpenAI embeddings
vectorstore = Chroma.from_texts(
    docs,
    embedding=OpenAIEmbeddings(),
)


In [20]:
# Create a retriever from the vector store to search for relevant documents
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

# Invoke the retriever with a query to find relevant documents
retriever.invoke("how social media data used to influence political opinions")

[Document(metadata={}, page_content='Social Media and Politics: The Cambridge Analytica scandal revealed how social media data could be used to influence political opinions. The company collected personal data from millions of Facebook users without consent and used it for political advertising. This event sparked a global discussion on data privacy and the ethical responsibilities of social media platforms.'),
 Document(metadata={}, page_content="Gene Editing: CRISPR-Cas9 technology has emerged as a groundbreaking tool in gene editing. It allows scientists to edit DNA sequences and modify gene function with high precision. Its potential applications range from treating genetic disorders to enhancing crop resilience. However, ethical debates arise around the possibilities of 'designer babies' and genetic privacy concerns.")]

In [21]:
template = """Answer the question based only on the following context:
{context}

Question: {question}
"""
prompt = ChatPromptTemplate.from_template(template)
model = ChatOpenAI()
output_parser = StrOutputParser()

In [22]:
# In case of multiple retrivers, we can use RunnableParallel to run the retriever and the model in parallel

# setup_and_retrieval = RunnableParallel(
#     {"context": retriever, "question": RunnablePassthrough()}
# )

In [23]:
# Create a chain that combines the retriever, prompt, model, and output parser

chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | model
    | output_parser
)

response = chain.invoke("how social media data used to influence political opinions")


In [24]:
pp(response)

('Social media data was used to influence political opinions by collecting '
 'personal data from millions of Facebook users without consent and using it '
 'for political advertising.')
